# Track 3 findings

Six questions, using saved AI responses and automatic measurements. No new API calls or sentiment scoring.

**Run:** install `track3/requirements_eda.txt`, prepare your input with `track3/prepare_eda.py`, set `BATCH_ID`, and run all cells. See the Track 3 README for commands. Use batch `sample` for the free synthetic demo.

**Read:** charts → findings → selected records. P01–P05 are unbranded; P06 names RPSD. Automatic labels and competitor-position estimates need review.


## Load the batch

One row = one response. Original text stays unchanged. Exports and figures stay in the private batch folder.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / 'track3/rpsd_rules.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Start the notebook inside the RPSD-Capstone repository.')
BATCH_ID = 'sample'  # change to your real batch after preparing its input
DATA_DIR = REPO / 'data' / 'track3' / 'eda' / BATCH_ID
INPUT_FILE = DATA_DIR / 'responses_input.csv'
OUTPUT_FILE = DATA_DIR / 'responses_for_sentiment.csv'

if not INPUT_FILE.exists():
    raise FileNotFoundError('Build this batch input first with track3/prepare_eda.py; see track3/README.md.')
df = pd.read_csv(INPUT_FILE, dtype=str, keep_default_na=False)
# Individual competitor positions are not present in the original labels.
# Preserve supplied values; otherwise leave blank for review (not zero or absent).
if 'competitor_positions' not in df.columns:
    df.insert(1, 'competitor_positions', '')
else:
    df.insert(1, 'competitor_positions', df.pop('competitor_positions'))

### Estimate competitor positions

Match saved competitor names to first appearances in the response. Unmatched names stay null; ambiguous matches are flagged. The function is defined here and runs with the notebook.

In [ ]:
"""Offline first-mention estimates from saved text and listed competitor names."""
import json
import re
import unicodedata

import sys
sys.path.insert(0, str(REPO / 'track3'))
from rpsd_rules import CANONICAL_CLINICS

ALIASES = dict(CANONICAL_CLINICS)
ALIASES.update({
    'CCRM': r'\bccrm\b',
    'CNY Fertility': r'\bcny fertility\b',
    'Kindbody San Diego': r'\bkindbody\b',
    'Kaiser Permanente San Diego': r'\bkaiser permanente\b',
    'Naval Medical Center San Diego': r'\bnaval medical cent(?:er|re)\b',
    'Sama Fertility': r'\bsama fertility\b',
})
RPSD_NAME = r'reproductive partners fertility cent(?:er|re)|\brpsd\b'


def estimate_positions(answer, competitors, rpsd_mentioned):
    """Order matched listed clinics; unmatched names stay null, not absent or zero.

    These are estimates among the saved competitor roster plus RPSD. They are
    not explicit recommendations, numbered headings, or validated clinic identity.
    """
    text = unicodedata.normalize('NFKC', str(answer)).lower()
    # Keep visible Markdown link labels but do not count clinic names inside URLs.
    text = re.sub(r'\]\(https?://[^)]+\)', ']', text)
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    names = list(dict.fromkeys(c.strip() for c in str(competitors).split(';') if c.strip()))
    candidates, notes = [], []
    for name in names:
        if 'reproductive partners medical group' in name.lower():
            notes.append(f'Ambiguous clinic identity: {name}')
            continue
        pattern = ALIASES.get(name, re.escape(unicodedata.normalize('NFKC', name).lower()))
        for match in re.finditer(pattern, text, re.I):
            candidates.append((match.start(), match.end(), name, match.group()))
    if rpsd_mentioned in ['Yes', 'Unclear']:
        for match in re.finditer(RPSD_NAME, text, re.I):
            candidates.append((match.start(), match.end(), 'RPSD', match.group()))
    # Resolve overlapping aliases by retaining the longer name, e.g. Southern
    # California Center... must not also count as California Center....
    matches = [m for m in candidates if not any(
        other[2] != m[2] and other[0] <= m[0] and other[1] >= m[1]
        and other[1]-other[0] > m[1]-m[0] for other in candidates)]
    first = {}
    for match in sorted(matches, key=lambda m: (m[0], -(m[1]-m[0]), m[2])):
        first.setdefault(match[2], match)
    ordered = sorted(first, key=lambda name: first[name][0])
    positions = {name: ordered.index(name)+1 if name in ordered else None for name in names}
    unmatched = [name for name in names if positions[name] is None]
    if unmatched:
        notes.append('Unmatched listed names: ' + '; '.join(unmatched))
    if rpsd_mentioned == 'Yes' and 'RPSD' not in first:
        notes.append('Saved RPSD mention has no matched target name')
    if rpsd_mentioned == 'Unclear':
        notes.append('Saved RPSD identity is unclear')
    if any(name in first for name in ['UC San Diego Health REI', 'CCRM']):
        notes.append('UCSD/CCRM alias requires clinic-identity review')
    if any('RPSD' == a[2] and a[0] == b[0] and b[2] != 'RPSD' for a in matches for b in matches):
        notes.append('Overlapping target and competitor identity')
    return {
        'auto_competitor_positions': json.dumps(positions, ensure_ascii=False),
        'auto_competitor_position_matches': json.dumps(
            {name: {'matched_text': first[name][3], 'position': positions[name]}
             for name in names if name in first}, ensure_ascii=False),
        'auto_competitor_position_review': '; '.join(notes),
        'auto_rpsd_position_from_text': ordered.index('RPSD')+1 if 'RPSD' in ordered else '',
    }


In [ ]:
position_estimates = pd.DataFrame([
    estimate_positions(row.response_text, row.auto_competitors, row.auto_rpsd_mentioned)
    for _, row in df.iterrows()
], index=df.index)
for column in position_estimates:
    df[column] = position_estimates[column]
df.insert(1, 'auto_competitor_positions', df.pop('auto_competitor_positions'))

original_responses = df.set_index('record_id')['response_text'].copy()
df['trial'] = pd.to_numeric(df['trial'])
pd.set_option('display.max_colwidth', 100)

from itertools import combinations
TABLES = DATA_DIR / 'research_tables'
TABLES.mkdir(exist_ok=True)
print(f'{len(df)} responses, {df.shape[1]} columns')
display(df[['provider','prompt_id','trial','auto_rpsd_mentioned']].head())
from IPython.display import Markdown

FIGURES = DATA_DIR / 'figures'
FIGURES.mkdir(exist_ok=True)
PROVIDERS = [p for p in ['OpenAI','Claude','Gemini'] if p in set(df.provider)]
COLORS = {'OpenAI':'#177e89','Claude':'#cf8542','Gemini':'#7961a8'}
FINDINGS = []
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False,
                     'axes.spines.right':False, 'font.size':10})

def show_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=160, bbox_inches='tight')
    plt.show()

def finding(question, text):
    FINDINGS[:] = [item for item in FINDINGS if item['question'] != question]
    FINDINGS.append({'question':question, 'finding':text})
    display(Markdown('**Finding:** ' + text))


## 1 Where does RPSD appear or disappear?

Rate = Yes / all observed trials, including Unclear. Missing trials are not No. P06 is shown separately.


In [ ]:
EXPECTED_TRIALS = {1, 2, 3}
df['mention_order'] = pd.to_numeric(df['auto_first_mention_order'], errors='coerce')
df['explicit_rank'] = pd.to_numeric(df['auto_explicit_rank'], errors='coerce')
question_results = df.groupby(['prompt_type','prompt_id','category','provider'], dropna=False).agg(
    responses=('record_id','size'),
    yes=('auto_rpsd_mentioned',lambda s:s.eq('Yes').sum()),
    no=('auto_rpsd_mentioned',lambda s:s.eq('No').sum()),
    unclear=('auto_rpsd_mentioned',lambda s:s.eq('Unclear').sum()),
    observed_trials=('trial',lambda s: ', '.join(str(int(v)) for v in sorted(s.dropna().unique()))),
).reset_index()
question_results['mention_rate'] = question_results['yes']/question_results['responses']
unbranded_results=question_results.loc[question_results.prompt_type.eq('Unbranded')]
display(df[['prompt_id','category','prompt_type']].drop_duplicates().sort_values('prompt_id'))


In [ ]:
if not unbranded_results.empty:
    rates=unbranded_results.pivot(index='prompt_id',columns='provider',values='mention_rate').reindex(columns=PROVIDERS)
    counts=unbranded_results.pivot(index='prompt_id',columns='provider',values='responses').reindex(columns=PROVIDERS)
    yes=unbranded_results.pivot(index='prompt_id',columns='provider',values='yes').reindex(columns=PROVIDERS)
    fig,axes=plt.subplots(1,2,figsize=(12,4.3))
    image=axes[0].imshow(rates,vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
    axes[0].set(xticks=range(len(PROVIDERS)),xticklabels=PROVIDERS,
                yticks=range(len(rates)),yticklabels=rates.index,title='Unbranded mention rate')
    for i in range(len(rates)):
        for j in range(len(PROVIDERS)):
            value=rates.iloc[i,j]
            label=f'{int(yes.iloc[i,j])}/{int(counts.iloc[i,j])}' if pd.notna(value) else 'Missing'
            axes[0].text(j,i,label,ha='center',va='center',color='white' if value>.6 else '#222')
    fig.colorbar(image,ax=axes[0],label='Yes / observed trials')
    totals=unbranded_results.groupby('provider')[['yes','no','unclear']].sum().reindex(PROVIDERS)
    totals.plot.bar(stacked=True,ax=axes[1],rot=0,color=['#177e89','#ced7dd','#e2aa4b'])
    axes[1].set(title='Unbranded presence counts',ylabel='Responses',xlabel='')
    axes[1].legend(['Yes','No','Unclear'],loc='upper left',bbox_to_anchor=(1,1))
    show_figure(fig,'01_mentions.png')
    question_totals=unbranded_results.groupby('prompt_id')[['yes','responses']].sum()
    question_totals['rate']=question_totals.yes/question_totals.responses
    low=question_totals.sort_values('rate').iloc[0]; low_id=question_totals.sort_values('rate').index[0]
    finding('Mentions',f'Lowest observed unbranded mention rate: {low_id}, {int(low.yes)}/{int(low.responses)} ({low.rate:.0%}). Ties are visible in the chart.')
branded=question_results.loc[question_results.prompt_type.eq('Branded')]
if not branded.empty:
    display(branded[['prompt_id','provider','yes','responses','unclear']])

## 2 Are the three trials consistent?

Pair agreement uses Yes/No labels only: Yes/Yes/No = 1/3; Yes/Yes/Yes = 3/3. Unclear pairs are excluded. High agreement can mean consistent absence.


In [ ]:
consistency_rows=[]
for (provider,prompt_id,prompt_type),group in df.groupby(['provider','prompt_id','prompt_type']):
    group=group.sort_values('trial')
    labels=group['auto_rpsd_mentioned'].tolist()
    known=[label for label in labels if label in ['Yes','No']]
    pairs=list(combinations(known,2))
    agreed=sum(a==b for a,b in pairs)
    observed=set(group.trial.dropna())
    consistency_rows.append({
        'provider':provider,'prompt_id':prompt_id,'prompt_type':prompt_type,
        'pattern': ' / '.join(labels),'responses':len(group),
        'complete_three_trials':observed==EXPECTED_TRIALS and len(group)==3,
        'unknown_or_invalid_trials':len(labels)-len(known),
        'agreeing_pairs':agreed,'known_pairs':len(pairs),
        'pair_agreement':agreed/len(pairs) if pairs else float('nan'),
    })
trial_consistency=pd.DataFrame(consistency_rows)
# Show missing slots against the full planned protocol, including absent groups.
expected_slots=pd.MultiIndex.from_product(
    [['OpenAI','Claude','Gemini'], ['P01','P02','P03','P04','P05','P06'], [1,2,3]],
    names=['provider','prompt_id','trial']).to_frame(index=False)
observed_slots=df[['provider','prompt_id','trial']].drop_duplicates()
missing_trial_slots=expected_slots.merge(observed_slots,on=['provider','prompt_id','trial'],how='left',indicator=True)
missing_trial_slots=missing_trial_slots.loc[missing_trial_slots['_merge'].eq('left_only')].drop(columns='_merge')
print(f'Missing trial slots versus full design: {len(missing_trial_slots)} / 54')


In [ ]:
unbranded_consistency=trial_consistency.loc[trial_consistency.prompt_type.eq('Unbranded')]
if not unbranded_consistency.empty:
    grid=unbranded_consistency.pivot(index='prompt_id',columns='provider',values='pair_agreement').reindex(columns=PROVIDERS)
    fig,ax=plt.subplots(figsize=(7.5,4))
    image=ax.imshow(grid,vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
    ax.set(xticks=range(len(PROVIDERS)),xticklabels=PROVIDERS,
           yticks=range(len(grid)),yticklabels=grid.index,title='Unbranded trial agreement')
    for i,prompt in enumerate(grid.index):
        for j,provider in enumerate(PROVIDERS):
            group=unbranded_consistency.loc[(unbranded_consistency.prompt_id==prompt)&(unbranded_consistency.provider==provider)]
            if group.empty: label='Missing'
            else:
                row=group.iloc[0]
                label=f'{int(row.agreeing_pairs)}/{int(row.known_pairs)} pairs' if row.known_pairs else 'No pairs'
                if row.unknown_or_invalid_trials:label+=' *'
            value=grid.iloc[i,j]
            ax.text(j,i,label,ha='center',va='center',color='white' if value>.6 else '#222',fontsize=9)
    fig.colorbar(image,ax=ax,label='Agreeing / known Yes-No pairs')
    ax.set_xlabel('* At least one unclear/invalid trial excluded')
    show_figure(fig,'02_trial_agreement.png')
complete=trial_consistency.loc[trial_consistency.complete_three_trials]
stable=complete.pair_agreement.eq(1)&complete.unknown_or_invalid_trials.eq(0)
finding('Consistency',f'{int(stable.sum())}/{len(complete)} complete provider/question groups have identical, unambiguous presence labels across all three trials (includes branded P06).')
changing=trial_consistency.loc[~trial_consistency.complete_three_trials|trial_consistency.unknown_or_invalid_trials.gt(0)|trial_consistency.pair_agreement.lt(1)]
display(changing[['provider','prompt_id','pattern','agreeing_pairs','known_pairs']])

## 3 How early is RPSD mentioned?

Use the automatic first-mention order for Yes responses; exclude missing orders. Order is not recommendation rank. Competitor positions are separate text-matching estimates, with unmatched names and review flags retained.


In [ ]:
position_rows=[]
for (provider,prompt_type),group in df.groupby(['provider','prompt_type']):
    mentioned=group.loc[group.auto_rpsd_mentioned.eq('Yes')]
    orders=mentioned.loc[mentioned.mention_order.gt(0),'mention_order']
    ranks=mentioned.loc[mentioned.explicit_rank.gt(0),'explicit_rank'] if prompt_type=='Unbranded' else pd.Series(dtype=float)
    position_rows.append({
        'provider':provider,'prompt_type':prompt_type,'yes_mentions':len(mentioned),
        'valid_orders':len(orders),'missing_or_invalid_orders':len(mentioned)-len(orders),
        'mean_order':orders.mean(),'median_order':orders.median(),
        'first_mentioned_count':orders.eq(1).sum(),
        'first_mentioned_rate':orders.eq(1).mean() if len(orders) else float('nan'),
        'explicitly_ranked_answers':len(ranks),'mean_explicit_rank':ranks.mean(),
    })
position_summary=pd.DataFrame(position_rows)

position_records=df.loc[df.auto_rpsd_mentioned.eq('Yes'),[
    'record_id','auto_competitor_positions','competitor_positions','auto_competitor_position_review','provider','prompt_id','prompt_type','auto_first_mention_order',
    'auto_numbered_list_position','auto_explicit_rank','auto_competitors']]


In [ ]:
eligible=df.loc[df.prompt_type.eq('Unbranded')&df.auto_rpsd_mentioned.eq('Yes')&df.mention_order.gt(0)]
if not eligible.empty:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    distribution=pd.crosstab(eligible.mention_order.astype(int),eligible.provider).reindex(columns=PROVIDERS,fill_value=0)
    distribution.plot.bar(ax=axes[0],rot=0,color=[COLORS[p] for p in PROVIDERS])
    axes[0].set(title='RPSD mention positions',xlabel='First-mention order',ylabel='Unbranded responses')
    axes[0].legend(fontsize=8)
    summary=position_summary.loc[position_summary.prompt_type.eq('Unbranded')].set_index('provider').reindex(PROVIDERS)
    axes[1].bar(PROVIDERS,summary.first_mentioned_rate,color=[COLORS[p] for p in PROVIDERS])
    axes[1].set(title='RPSD appears first, conditional on mention',ylabel='First / valid orders',ylim=(0,1.1))
    for i,(_,row) in enumerate(summary.iterrows()):
        if row.valid_orders:axes[1].text(i,row.first_mentioned_rate+.025,f'{int(row.first_mentioned_count)}/{int(row.valid_orders)}',ha='center')
    show_figure(fig,'03_positions.png')
    finding('Position','; '.join(f'{p}: first in {int(r.first_mentioned_count)}/{int(r.valid_orders)} unbranded answers with valid RPSD orders' for p,r in summary.iterrows() if r.valid_orders)+'.')
display(position_summary[['provider','prompt_type','valid_orders','mean_order','explicitly_ranked_answers','mean_explicit_rank']].round(2))

## 4 Which competitors appear alongside RPSD?

Count each competitor once per response. Rates use RPSD-mentioned responses as the denominator. Competitors can overlap, so rates do not sum to 100%.


In [ ]:
co_rows=[]
for _,row in df.loc[df.auto_rpsd_mentioned.eq('Yes')].iterrows():
    competitors={c.strip() for c in row.auto_competitors.split(';') if c.strip()}
    for competitor in competitors:
        co_rows.append({'record_id':row.record_id,'provider':row.provider,
                        'prompt_type':row.prompt_type,'competitor':competitor})
competitor_mentions=pd.DataFrame(co_rows,columns=['record_id','provider','prompt_type','competitor'])
denominators=df.loc[df.auto_rpsd_mentioned.eq('Yes')].groupby(['provider','prompt_type']).size().rename('rpsd_answers')
competitor_summary=(competitor_mentions.groupby(['provider','prompt_type','competitor']).size()
                    .rename('co_mentions').reset_index())
competitor_summary=competitor_summary.merge(denominators.reset_index(),on=['provider','prompt_type'],how='left')
competitor_summary['co_mention_rate']=competitor_summary.co_mentions/competitor_summary.rpsd_answers
competitor_summary=competitor_summary.sort_values(['prompt_type','co_mentions'],ascending=[True,False])


In [ ]:
co=competitor_mentions.loc[competitor_mentions.prompt_type.eq('Unbranded')]
if not co.empty:
    top=co.groupby('competitor').size().sort_values(ascending=False).head(8)
    denominator=int((df.prompt_type.eq('Unbranded')&df.auto_rpsd_mentioned.eq('Yes')).sum())
    estimated=[]
    for _,row in df.loc[df.prompt_type.eq('Unbranded')&df.auto_rpsd_mentioned.eq('Yes')].iterrows():
        for clinic,position in json.loads(row.auto_competitor_positions).items():
            if position is not None:estimated.append({'competitor':clinic,'position':position})
    estimated=pd.DataFrame(estimated,columns=['competitor','position'])
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    top.sort_values().plot.barh(ax=axes[0],color='#177e89')
    axes[0].set(title=f'Competitor co-mentions (RPSD answers n={denominator})',xlabel='Responses',ylabel='')
    for i,value in enumerate(top.sort_values()):axes[0].text(value+.1,i,f'{value}/{denominator}',va='center',fontsize=8)
    axes[0].set_xlim(0,top.max()*1.27)
    stats=estimated.loc[estimated.competitor.isin(top.index)].groupby('competitor').position.agg(['mean','count']).sort_values('mean',ascending=False)
    if not stats.empty:
        axes[1].barh(stats.index,stats['mean'],color='#7961a8')
        for i,(_,row) in enumerate(stats.iterrows()):axes[1].text(row['mean']+.05,i,f'n={int(row["count"])}',va='center',fontsize=8)
        axes[1].set_xlim(0,stats['mean'].max()*1.3)
    axes[1].set(title='Estimated competitor positions',xlabel='Mean first-mention position; matched names only',ylabel='')
    show_figure(fig,'04_competitors.png')
    finding('Competitors',f'{top.index[0]} is a most frequent co-mentioned competitor: {int(top.iloc[0])}/{denominator} unbranded RPSD-mentioned responses. Estimated positions require review.')
else:
    finding('Competitors','No competitor co-mentions in unbranded RPSD-mentioned responses.')

## 5 Which claim types need qualification?

Verdicts come from the saved fact-check evidence. Topics use visible keyword rules and are provisional. Counts are claim occurrences across trials, not unique facts. “Not documented” does not mean false.


In [ ]:
TOPIC_RULES=[
    ('Insurance / cost', r'insurance|aetna|anthem|cigna|unitedhealth|health ?net|progyny|tricare|ppo|pos\b|cost|financ|coverage'),
    ('Physicians / availability', r'physician|doctor|dr\.|rotate|rotation|rarely|meldrum|hsieh|garzo|whynott|duleba|agarwal'),
    ('UCSD affiliation / ownership', r'ucsd|uc san diego|university of california|affiliat|ownership|owned'),
    ('Services / laboratory', r'ivf|iui|freez|donor|surrog|genetic|laboratory|embry|fertility testing'),
    ('Location / contact', r'address|located|location|la jolla|phone|website|hours|genesee'),
]
# First matching topic rule wins; edit these visible rules if needed.
claim_rows=[]
for _,row in df.iterrows():
    for line in row.auto_fact_check_evidence.splitlines():
        match=re.match(r'^(SUPPORTED|NEEDS QUALIFICATION|CONTRADICTED|NOT DOCUMENTED): (.+)$',line)
        if not match:continue
        verdict,body=match.groups()
        claim_text=body.split(' - ',1)[0]
        claim_text=re.sub(r'\s*\[F\d+[^\]]*\]','',claim_text).strip()
        topic=next((name for name,pattern in TOPIC_RULES if re.search(pattern,claim_text,re.I)),'Other / review')
        claim_rows.append({'record_id':row.record_id,'provider':row.provider,'prompt_id':row.prompt_id,
                           'claim_text':claim_text,'verdict':verdict,'provisional_topic':topic,'evidence_line':line})
claims=pd.DataFrame(claim_rows,columns=['record_id','provider','prompt_id','claim_text','verdict','provisional_topic','evidence_line'])
claim_summary=claims.groupby('provisional_topic').agg(
    claim_occurrences=('verdict','size'), distinct_answers=('record_id','nunique'),
    needs_qualification=('verdict',lambda s:s.eq('NEEDS QUALIFICATION').sum()),
    supported=('verdict',lambda s:s.eq('SUPPORTED').sum()),
    contradicted=('verdict',lambda s:s.eq('CONTRADICTED').sum()),
    not_documented=('verdict',lambda s:s.eq('NOT DOCUMENTED').sum()),
).reset_index()
claim_summary['qualification_rate']=claim_summary.needs_qualification/claim_summary.claim_occurrences


In [ ]:
if not claim_summary.empty:
    summary=claim_summary.sort_values('qualification_rate')
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    counts=summary.set_index('provisional_topic')[['supported','needs_qualification','not_documented','contradicted']]
    counts.plot.barh(stacked=True,ax=axes[0],color=['#177e89','#e2aa4b','#cbd4da','#b94d61'])
    axes[0].set(title='Saved claim verdicts by provisional topic',xlabel='Claim occurrences',ylabel='')
    axes[0].legend(['Supported','Needs qualification','Not documented','Contradicted'],fontsize=8,loc='upper left',bbox_to_anchor=(0,-.15),ncol=2)
    axes[1].barh(summary.provisional_topic,summary.qualification_rate,color='#cf8542')
    axes[1].set(title='Qualification rate',xlabel='Needs qualification / all topic claims',xlim=(0,1.12))
    for i,(_,row) in enumerate(summary.iterrows()):
        axes[1].text(row.qualification_rate+.02,i,f'{int(row.needs_qualification)}/{int(row.claim_occurrences)}',va='center')
    show_figure(fig,'05_claims.png')
    high=summary.iloc[-1]
    finding('Claims',f'Highest provisional topic qualification rate: {high.provisional_topic}, {int(high.needs_qualification)}/{int(high.claim_occurrences)} claim occurrences ({high.qualification_rate:.0%}), across {int(high.distinct_answers)} answers. Topics are keyword-based and repeated trials repeat claims.')
display(claim_summary[['provisional_topic','claim_occurrences','distinct_answers','needs_qualification','qualification_rate']].round(3))

## 6 What needs human review?

Flags prioritize inspection; they are not confirmed errors. A response may have several flags. Missing inline citations alone is not flagged because retrieved sources may exist.


In [ ]:
review_flags=[]
for _,row in df.iterrows():
    reasons=[]
    if row.auto_competitor_position_review:reasons.append('Competitor position: '+row.auto_competitor_position_review)
    if row.auto_rpsd_mentioned=='Yes' and row.auto_rpsd_position_from_text != '' and row.mention_order>0 and row.auto_rpsd_position_from_text != row.mention_order:
        reasons.append('Text-estimated RPSD position differs from saved mention order')
    if row.auto_rpsd_mentioned not in ['Yes','No','Unclear']:reasons.append('Missing/invalid mention label')
    if row.auto_rpsd_mentioned=='Unclear':reasons.append('Ambiguous RPSD identity')
    if row.auto_rpsd_mentioned=='Yes' and not row.mention_order>0:reasons.append('Mentioned but no valid mention order')
    if row.auto_rpsd_mentioned=='No' and row.mention_order>0:reasons.append('Absent but numeric mention order')
    if any(word in row.auto_accuracy_assessment.lower() for word in ['qualification','contradict']):reasons.append('Claim qualification/contradiction needs review')
    if re.search(r'claim\(s\) dropped|Identity:|Judge rank',row.auto_label_notes,re.I):reasons.append('Extraction/identity/rank normalization note')
    for field in ['response_text','model_returned','search_setting','collected_at_utc']:
        if not str(row[field]).strip():reasons.append('Missing '+field)
    if pd.isna(pd.to_datetime(row.collected_at_utc,utc=True,errors='coerce')):reasons.append('Invalid timestamp')
    if df.record_id.eq(row.record_id).sum()>1:reasons.append('Duplicate record ID')
    if df.response_text.eq(row.response_text).sum()>1:reasons.append('Repeated identical answer text')
    if (df.provider.eq(row.provider)&df.prompt_id.eq(row.prompt_id)&df.trial.eq(row.trial)).sum()>1:reasons.append('Duplicate provider/question/trial slot')
    if reasons:
        review_flags.append({'record_id':row.record_id,'provider':row.provider,'prompt_id':row.prompt_id,
                              'review_reasons':'; '.join(reasons)})
review_queue=pd.DataFrame(review_flags,columns=['record_id','provider','prompt_id','review_reasons'])


In [ ]:
flag_patterns={
    'Competitor-position estimate':r'Competitor position:|Text-estimated RPSD position',
    'Claim qualification / contradiction':r'Claim qualification/contradiction',
    'Extraction / rank / identity note':r'Extraction/identity/rank',
    'Ambiguous target identity':r'Ambiguous RPSD identity',
    'Missing / inconsistent metadata':r'Missing|Invalid timestamp|numeric mention order|no valid mention order',
    'Duplicate records or text':r'Duplicate|Repeated identical',
}
flag_counts=pd.Series({name:int(review_queue.review_reasons.str.contains(pattern,regex=True).sum())
                      for name,pattern in flag_patterns.items()}).sort_values()
flagged_by_provider=review_queue.groupby('provider').record_id.nunique().reindex(PROVIDERS,fill_value=0)
totals=df.groupby('provider').size().reindex(PROVIDERS)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
flag_counts.plot.barh(ax=axes[0],color='#cf8542')
axes[0].set(title='Review reasons (can overlap)',xlabel='Flagged responses',ylabel='')
axes[1].bar(PROVIDERS,flagged_by_provider/totals,color=[COLORS[p] for p in PROVIDERS])
axes[1].set(title='Responses with any review flag',ylabel='Flagged / collected',ylim=(0,1.14))
for i,p in enumerate(PROVIDERS):axes[1].text(i,flagged_by_provider[p]/totals[p]+.025,f'{flagged_by_provider[p]}/{totals[p]}',ha='center')
show_figure(fig,'06_review.png')
finding('Review',f'{len(review_queue)}/{len(df)} responses have at least one review flag. This is a triage count, not an error rate; some flags are expected qualifications or alias ambiguity.')
display(review_queue.head(10))

# Call this with an ID from the review queue when you want the full evidence.
def inspect_response(record_id):
    row=df.loc[df.record_id.eq(record_id)].iloc[0]
    print('QUESTION:',row.prompt_text,'\n\nRESPONSE:\n',row.response_text)
    print('\nLABEL NOTES:\n',row.auto_label_notes)
    print('\nFACT CHECK:\n',row.auto_fact_check_evidence)

# inspect_response('siya-pilot-01:API-OpenAI-P01-1')

## Findings at a glance

Generated from this batch. Review flagged evidence before using these statements in the report.

In [ ]:
findings_table=pd.DataFrame(FINDINGS)
with pd.option_context('display.max_colwidth', None):
    display(findings_table)
findings_table.to_csv(TABLES/'findings.csv',index=False,encoding='utf-8-sig')

## Save findings and handoff

Research tables include counts, denominators, and record IDs. The Track 2 handoff retains all original responses and excludes sentiment output columns. Rerunning replaces generated outputs only.


In [ ]:
tables={'question_mentions':question_results,'trial_consistency':trial_consistency,'missing_trial_slots':missing_trial_slots,
        'mention_positions':position_summary,'position_records':position_records,
        'competitor_co_mentions':competitor_summary,'competitor_records':competitor_mentions,
        'claim_topics':claim_summary,'claim_records':claims,'review_queue':review_queue}
for name,table in tables.items():
    table.to_csv(TABLES/f'{name}.csv',index=False,encoding='utf-8-sig')
assert df.record_id.is_unique
assert set(df.record_id)==set(original_responses.index)
assert df.set_index('record_id').response_text.reindex(original_responses.index).equals(original_responses)
track2_outputs={'sentiment_label','sentiment_score','prob_negative','prob_neutral','prob_positive'}
assert not track2_outputs.intersection(df.columns)
# Numeric helper columns are analysis-only, not part of the handoff schema.
df.drop(columns=['mention_order','explicit_rank']).to_csv(OUTPUT_FILE,index=False,encoding='utf-8-sig')
print(f'Saved {len(tables)} research tables to {TABLES}')
print(f'Saved {len(df)} unchanged responses to {OUTPUT_FILE.name}')